# P2 — Clasificación del desequilibrio horario por estación en MiBici

**Objetivo:** construir y evaluar un clasificador que estime el nivel de desequilibrio de una estación para la siguiente hora, usando únicamente datos históricos de viajes y variables temporales.

Este notebook está dedicado exclusivamente a la **Pregunta 2 (P2)**: lectura de datos, limpieza, construcción de flujos estación-hora, generación de variables y objetivo, partición temporal, entrenamiento, evaluación y seguimiento de experimentos con MLflow.

> **Interpretación:** las clases `bajo`, `medio` y `alto` representan desequilibrio observado entre llegadas y salidas; no equivalen a una medición directa de inventario de bicicletas ni a una garantía de riesgo operativo real.

## 1. Configuración y lectura de datos

In [ ]:
from pathlib import Path
import hashlib
import json
import tempfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Permite ejecutar desde la raíz del repositorio o desde notebooks/.
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "data").exists() else CWD.parent
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", ROOT)
print("Datos raw:", DATA_RAW)
print("Datos procesados:", DATA_PROCESSED)

In [ ]:
# El repositorio debe contener un único CSV de viajes en data/raw.
archivos_csv = sorted(DATA_RAW.glob("*.csv"))
if not archivos_csv:
    raise FileNotFoundError(
        f"No se encontraron CSV en {DATA_RAW}. Coloca ahí el archivo de viajes de MiBici."
    )
if len(archivos_csv) > 1:
    candidatos = [p.name for p in archivos_csv]
    raise ValueError(
        f"Hay varios CSV en {DATA_RAW}: {candidatos}. Ajusta TRIPS_PATH para elegir uno."
    )

TRIPS_PATH = archivos_csv[0]
print(f"Archivo: {TRIPS_PATH.name} ({TRIPS_PATH.stat().st_size / 1024**2:.1f} MB)")

columnas_requeridas = [
    "Inicio_del_viaje", "Fin_del_viaje", "Origen_Id", "Destino_Id"
]
trips = pd.read_csv(
    TRIPS_PATH,
    usecols=columnas_requeridas,
    dtype={"Origen_Id": "Int32", "Destino_Id": "Int32"},
    parse_dates=["Inicio_del_viaje", "Fin_del_viaje"],
    low_memory=False,
)

faltantes = set(columnas_requeridas) - set(trips.columns)
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")

print(f"Filas: {len(trips):,}")
print("Rango de inicio:", trips["Inicio_del_viaje"].min(), "a", trips["Inicio_del_viaje"].max())
print("Memoria aproximada (MB):", round(trips.memory_usage(deep=True).sum()/1024**2, 1))
display(trips.head())

## 2. Limpieza de viajes para P2
Se excluyen duraciones de un minuto o menos, duraciones de tres horas o más, fechas inválidas y viajes de ida y vuelta con la misma estación. Se conserva solo la información necesaria para construir llegadas y salidas.

In [ ]:
n_original = len(trips)
trips_limpios = trips.copy()
trips_limpios["duracion_min"] = (
    trips_limpios["Fin_del_viaje"] - trips_limpios["Inicio_del_viaje"]
).dt.total_seconds() / 60

mascara_valida = (
    trips_limpios["Inicio_del_viaje"].notna()
    & trips_limpios["Fin_del_viaje"].notna()
    & trips_limpios["Origen_Id"].notna()
    & trips_limpios["Destino_Id"].notna()
    & (trips_limpios["duracion_min"] > 1)
    & (trips_limpios["duracion_min"] < 180)
    & (trips_limpios["Origen_Id"] != trips_limpios["Destino_Id"])
)
trips_limpios = trips_limpios.loc[mascara_valida].copy()
n_limpios = len(trips_limpios)

resumen_limpieza = pd.DataFrame({
    "indicador": ["Viajes originales", "Viajes conservados", "Viajes excluidos", "% conservado", "% excluido"],
    "valor": [n_original, n_limpios, n_original-n_limpios,
              round(100*n_limpios/n_original, 2), round(100*(n_original-n_limpios)/n_original, 2)]
})
display(resumen_limpieza)
display(trips_limpios["duracion_min"].describe().to_frame("duracion_min"))

## 3. Construcción del flujo por estación y hora
Las salidas se asignan a la hora de inicio y estación de origen; las llegadas, a la hora de finalización y estación de destino. El flujo neto se define como `llegadas - salidas`. Como la fuente no registra explícitamente el inventario disponible, el flujo neto se utiliza como aproximación de desequilibrio.

In [ ]:
salidas = (
    trips_limpios.assign(hora=trips_limpios["Inicio_del_viaje"].dt.floor("h"))
    .groupby(["Origen_Id", "hora"], observed=True).size()
    .rename("salidas").reset_index()
    .rename(columns={"Origen_Id": "estacion_id"})
)
llegadas = (
    trips_limpios.assign(hora=trips_limpios["Fin_del_viaje"].dt.floor("h"))
    .groupby(["Destino_Id", "hora"], observed=True).size()
    .rename("llegadas").reset_index()
    .rename(columns={"Destino_Id": "estacion_id"})
)

flujo_horario = salidas.merge(llegadas, on=["estacion_id", "hora"], how="outer")
flujo_horario[["salidas", "llegadas"]] = flujo_horario[["salidas", "llegadas"]].fillna(0).astype("int32")
flujo_horario["flujo_neto"] = flujo_horario["llegadas"] - flujo_horario["salidas"]
flujo_horario = flujo_horario.sort_values(["estacion_id", "hora"]).reset_index(drop=True)

print(f"Registros estación-hora: {len(flujo_horario):,}")
print(f"Estaciones: {flujo_horario['estacion_id'].nunique():,}")
print(f"Horas distintas con actividad: {flujo_horario['hora'].nunique():,}")
display(flujo_horario.head())

## 4. Definición de clases y variables predictoras
Las clases se definen con umbrales operativos provisionales: `bajo` para flujo neto entre −3 y 3; `medio` para magnitudes 4 o 5; `alto` para magnitudes iguales o superiores a 6. Las variables históricas se aceptan únicamente cuando el registro anterior de esa estación corresponde exactamente a una hora antes.

In [ ]:
def clasificar_desequilibrio(flujo):
    if -3 <= flujo <= 3:
        return "bajo"
    if -5 <= flujo <= -4 or 4 <= flujo <= 5:
        return "medio"
    return "alto"

flujo_horario["riesgo_provisional"] = flujo_horario["flujo_neto"].map(clasificar_desequilibrio)

# Variables temporales y variables históricas por estación.
grupos = flujo_horario.groupby("estacion_id", sort=False)
flujo_horario["hora_anterior"] = grupos["hora"].shift(1)
flujo_horario["salidas_hora_anterior"] = grupos["salidas"].shift(1)
flujo_horario["llegadas_hora_anterior"] = grupos["llegadas"].shift(1)
flujo_horario["flujo_neto_hora_anterior"] = grupos["flujo_neto"].shift(1)

continuidad_anterior = (
    (flujo_horario["hora"] - flujo_horario["hora_anterior"]).dt.total_seconds().div(3600) == 1
)
for col in ["salidas_hora_anterior", "llegadas_hora_anterior", "flujo_neto_hora_anterior"]:
    flujo_horario.loc[~continuidad_anterior, col] = np.nan

flujo_horario["dia_semana"] = flujo_horario["hora"].dt.dayofweek
flujo_horario["mes"] = flujo_horario["hora"].dt.month
hora_dia = flujo_horario["hora"].dt.hour
flujo_horario["hora_sin"] = np.sin(2*np.pi*hora_dia/24)
flujo_horario["hora_cos"] = np.cos(2*np.pi*hora_dia/24)
flujo_horario["flujo_neto_anterior"] = flujo_horario["flujo_neto_hora_anterior"]
flujo_horario["salidas_anterior"] = flujo_horario["salidas_hora_anterior"]
flujo_horario["llegadas_anterior"] = flujo_horario["llegadas_hora_anterior"]

# Etiqueta objetivo: clase de la siguiente observación solo si está a una hora exacta.
flujo_horario["hora_siguiente"] = grupos["hora"].shift(-1)
flujo_horario["riesgo_siguiente"] = grupos["riesgo_provisional"].shift(-1)
diferencia_siguiente = (flujo_horario["hora_siguiente"] - flujo_horario["hora"]).dt.total_seconds().div(3600)
flujo_horario.loc[diferencia_siguiente != 1, "riesgo_siguiente"] = np.nan

FEATURES = ["dia_semana", "mes", "hora_sin", "hora_cos", "flujo_neto_anterior", "salidas_anterior", "llegadas_anterior"]
TARGET = "riesgo_siguiente"

print("Distribución de la clase actual:")
display(flujo_horario["riesgo_provisional"].value_counts(normalize=True).mul(100).round(2).rename("porcentaje").to_frame())
print("Filas con etiqueta futura válida:", f"{flujo_horario[TARGET].notna().sum():,}")
display(flujo_horario[["estacion_id", "hora", "flujo_neto", "riesgo_provisional", *FEATURES, TARGET]].head(10))

## 5. Partición temporal y persistencia de datos preparados
Se reserva julio–septiembre de 2025 para validación y octubre–diciembre de 2025 para prueba. Los datos anteriores a julio de 2025 se usan para entrenamiento. La selección de modelo se hace con validación; el conjunto de prueba se usa al final.

In [ ]:
datos_con_objetivo = flujo_horario.loc[flujo_horario[TARGET].notna()].copy()
train_df = datos_con_objetivo[datos_con_objetivo["hora"] < pd.Timestamp("2025-07-01")].copy()
valid_df = datos_con_objetivo[(datos_con_objetivo["hora"] >= pd.Timestamp("2025-07-01")) & (datos_con_objetivo["hora"] < pd.Timestamp("2025-10-01"))].copy()
test_df = datos_con_objetivo[(datos_con_objetivo["hora"] >= pd.Timestamp("2025-10-01")) & (datos_con_objetivo["hora"] < pd.Timestamp("2026-01-01"))].copy()

train_model = train_df.dropna(subset=FEATURES + [TARGET]).copy()
valid_model = valid_df.dropna(subset=FEATURES + [TARGET]).copy()
test_model = test_df.dropna(subset=FEATURES + [TARGET]).copy()

particiones = []
for nombre, df in [("train", train_model), ("validación", valid_model), ("test", test_model)]:
    particiones.append({"conjunto": nombre, "registros": len(df), "inicio": df["hora"].min(), "fin": df["hora"].max()})
display(pd.DataFrame(particiones))

for nombre, df in [("p2_train.parquet", train_model), ("p2_valid.parquet", valid_model), ("p2_test.parquet", test_model)]:
    df.to_parquet(DATA_PROCESSED / nombre, index=False)
    print(f"Guardado {nombre}: {len(df):,} filas")

# Huella del CSV fuente para trazabilidad.
def sha256_file(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_SHA256 = sha256_file(TRIPS_PATH)
print("SHA-256 datos fuente:", DATA_SHA256)

X_train, y_train = train_model[FEATURES], train_model[TARGET]
X_valid, y_valid = valid_model[FEATURES], valid_model[TARGET]
X_test, y_test = test_model[FEATURES], test_model[TARGET]
print("Dimensiones X_train/X_valid/X_test:", X_train.shape, X_valid.shape, X_test.shape)

## 6. Entrenamiento y comparación de clasificadores
La métrica principal es **F1 macro**, que da el mismo peso a cada clase. Se usa **balanced accuracy** como métrica complementaria debido al desbalance entre clases. La clase `alto` se evalúa también mediante recall y precisión, porque interesa detectar episodios de mayor desequilibrio.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

LABELS = ["bajo", "medio", "alto"]

def metricas_clasificacion(y_real, y_pred, sufijo="valid"):
    return {
        f"balanced_accuracy_{sufijo}": float(balanced_accuracy_score(y_real, y_pred)),
        f"f1_macro_{sufijo}": float(f1_score(y_real, y_pred, average="macro")),
        f"precision_alto_{sufijo}": float(precision_score(y_real, y_pred, labels=["alto"], average="macro", zero_division=0)),
        f"recall_alto_{sufijo}": float(recall_score(y_real, y_pred, labels=["alto"], average="macro", zero_division=0)),
        f"f1_alto_{sufijo}": float(f1_score(y_real, y_pred, labels=["alto"], average="macro", zero_division=0)),
    }

modelo_logreg = make_pipeline(
    SimpleImputer(strategy="median"), StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
modelo_rf_v1 = RandomForestClassifier(
    n_estimators=150, max_depth=15, min_samples_leaf=20,
    class_weight="balanced", random_state=42, n_jobs=-1
)
modelo_rf_v2 = RandomForestClassifier(
    n_estimators=150, max_depth=20, min_samples_leaf=10,
    class_weight="balanced", random_state=42, n_jobs=-1
)

modelos = {
    "LogisticRegression": modelo_logreg,
    "RandomForest_v1": modelo_rf_v1,
    "RandomForest_v2": modelo_rf_v2,
}

predicciones_validacion = {}
resultados_validacion = []
for nombre, modelo in modelos.items():
    print(f"Entrenando {nombre}...")
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_valid)
    predicciones_validacion[nombre] = pred
    fila = {"modelo": nombre, **metricas_clasificacion(y_valid, pred, "valid")}
    resultados_validacion.append(fila)

tabla_validacion = pd.DataFrame(resultados_validacion).sort_values("f1_macro_valid", ascending=False)
display(tabla_validacion.round(4))

### 6.1 Experimento adicional con HistGradientBoosting
Para evaluar el costo de entrenamiento, este modelo se ajusta sobre una muestra reproducible de 300,000 filas. Sus resultados son orientativos y no constituyen una comparación de costo idéntico frente a los modelos entrenados con todo el conjunto.

In [ ]:
n_muestra = min(300_000, len(train_model))
muestra_train = train_model.sample(n=n_muestra, random_state=42)
modelo_hgb = HistGradientBoostingClassifier(
    max_iter=100, max_leaf_nodes=31, learning_rate=0.1,
    random_state=42, class_weight="balanced"
)
modelo_hgb.fit(muestra_train[FEATURES], muestra_train[TARGET])
y_valid_pred_hgb = modelo_hgb.predict(X_valid)
predicciones_validacion["HistGradientBoosting_muestra"] = y_valid_pred_hgb
resultados_validacion.append({
    "modelo": "HistGradientBoosting_muestra",
    **metricas_clasificacion(y_valid, y_valid_pred_hgb, "valid")
})
tabla_validacion = pd.DataFrame(resultados_validacion).sort_values("f1_macro_valid", ascending=False)
display(tabla_validacion.round(4))

## 7. Selección y evaluación final en test
Se selecciona el modelo por F1 macro en validación. La evaluación de test se ejecuta una sola vez después de esa selección. Si se desea mantener exactamente la decisión documentada en los experimentos anteriores, puede fijarse `RandomForest_v1`; el criterio por defecto de esta celda selecciona el mejor F1 macro medido en validación.

In [ ]:
mejor_nombre = tabla_validacion.iloc[0]["modelo"]
modelos_disponibles = {**modelos, "HistGradientBoosting_muestra": modelo_hgb}
modelo_final = modelos_disponibles[mejor_nombre]
print("Modelo seleccionado por F1 macro de validación:", mejor_nombre)

y_test_pred = modelo_final.predict(X_test)
metricas_test = metricas_clasificacion(y_test, y_test_pred, "test")
print("Métricas de test:")
display(pd.DataFrame([metricas_test]).round(4))
print("Reporte por clase:")
print(classification_report(y_test, y_test_pred, labels=LABELS, digits=4, zero_division=0))
cm = confusion_matrix(y_test, y_test_pred, labels=LABELS)
display(pd.DataFrame(cm, index=[f"real_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS]))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_xticks(range(len(LABELS)), LABELS)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("Predicción")
ax.set_ylabel("Valor real")
ax.set_title(f"Matriz de confusión — {mejor_nombre}")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
fig.tight_layout()
plt.show()

## 8. Seguimiento de experimentos con MLflow
Se registra cada modelo de validación y el modelo seleccionado en test. Cada run incluye parámetros, métricas, etiquetas de trazabilidad y artefactos de evaluación. MLflow guarda su base SQLite en `mlruns/mlflow.db` dentro de la raíz del repositorio y los artefactos en el directorio local de seguimiento.

In [ ]:
import mlflow
import mlflow.sklearn
import sklearn

MLFLOW_DIR = ROOT / "mlruns"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DB = MLFLOW_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
EXPERIMENT_NAME = "MiBici_P2_Clasificacion_Riesgo"
mlflow.set_experiment(EXPERIMENT_NAME)

# Si Git está disponible, registrar el commit actual sin detener el notebook si no lo está.
import subprocess
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    GIT_COMMIT = "no_disponible"

print("MLflow:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", EXPERIMENT_NAME)
print("SHA-256 datos:", DATA_SHA256)
print("Git commit:", GIT_COMMIT)

In [ ]:
# Registrar experimentos de validación: parámetros, métricas, reporte y matriz de confusión.
modelos_para_mlflow = {
    "LogisticRegression_validacion": (modelo_logreg, y_valid, predicciones_validacion["LogisticRegression"], "LogisticRegression"),
    "RandomForest_v1_validacion": (modelo_rf_v1, y_valid, predicciones_validacion["RandomForest_v1"], "RandomForest"),
    "RandomForest_v2_validacion": (modelo_rf_v2, y_valid, predicciones_validacion["RandomForest_v2"], "RandomForest"),
    "HistGradientBoosting_validacion": (modelo_hgb, y_valid, predicciones_validacion["HistGradientBoosting_muestra"], "HistGradientBoosting"),
}

for run_name, (modelo, y_real, y_pred, tipo_modelo) in modelos_para_mlflow.items():
    metricas = metricas_clasificacion(y_real, y_pred, "valid")
    reporte = classification_report(y_real, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    cm_valid = confusion_matrix(y_real, y_pred, labels=LABELS)
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "classification_report.json").write_text(json.dumps(reporte, indent=2), encoding="utf-8")
        pd.DataFrame(cm_valid, index=LABELS, columns=LABELS).to_csv(tmp / "confusion_matrix.csv", index=True)
        pd.DataFrame([metricas]).to_csv(tmp / "metrics.csv", index=False)
        with mlflow.start_run(run_name=run_name):
            mlflow.log_param("modelo", tipo_modelo)
            mlflow.log_param("features", ",".join(FEATURES))
            mlflow.log_param("target", TARGET)
            mlflow.log_param("n_train", len(train_model))
            mlflow.log_param("n_valid", len(valid_model))
            mlflow.log_param("n_test", len(test_model))
            mlflow.log_param("data_sha256", DATA_SHA256)
            mlflow.log_param("git_commit", GIT_COMMIT)
            mlflow.log_param("split_train_hasta", "2025-06-30")
            mlflow.log_param("split_validacion", "2025-07-01/2025-09-30")
            mlflow.log_param("split_test", "2025-10-01/2025-12-31")
            if hasattr(modelo, "get_params"):
                mlflow.log_params({f"model_{k}": str(v) for k, v in modelo.get_params().items()})
            mlflow.log_metrics(metricas)
            mlflow.set_tags({"pregunta": "P2", "dataset": TRIPS_PATH.name, "fase": "validacion", "equipo": "37"})
            mlflow.log_artifacts(str(tmp), artifact_path="evaluation")
    print(f"Registrado: {run_name} | F1 macro={metricas['f1_macro_valid']:.4f} | Balanced accuracy={metricas['balanced_accuracy_valid']:.4f}")

In [ ]:
# Run de test para el modelo elegido, manteniendo explícito que test no participa en selección.
reporte_test = classification_report(y_test, y_test_pred, labels=LABELS, output_dict=True, zero_division=0)
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    (tmp / "classification_report_test.json").write_text(json.dumps(reporte_test, indent=2), encoding="utf-8")
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(tmp / "confusion_matrix_test.csv", index=True)
    pd.DataFrame([metricas_test]).to_csv(tmp / "metrics_test.csv", index=False)
    with mlflow.start_run(run_name=f"{mejor_nombre}_test"):
        mlflow.log_param("modelo_seleccionado", mejor_nombre)
        mlflow.log_param("features", ",".join(FEATURES))
        mlflow.log_param("target", TARGET)
        mlflow.log_param("data_sha256", DATA_SHA256)
        mlflow.log_param("git_commit", GIT_COMMIT)
        mlflow.log_param("criterio_seleccion", "F1 macro en validación")
        mlflow.log_metrics(metricas_test)
        mlflow.set_tags({"pregunta": "P2", "dataset": TRIPS_PATH.name, "fase": "test_final", "equipo": "37"})
        mlflow.log_artifacts(str(tmp), artifact_path="evaluation")
        # El modelo queda registrado como artefacto para reproducibilidad.
        mlflow.sklearn.log_model(modelo_final, artifact_path="model")
print("Run final de test registrado.")

## 9. Consultar los runs de P2 en MLflow

In [ ]:
experimento = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    output_format="pandas",
    order_by=["attributes.start_time DESC"],
)
columnas_prioritarias = [
    "tags.mlflow.runName", "tags.fase", "tags.pregunta",
    "metrics.f1_macro_valid", "metrics.balanced_accuracy_valid",
    "metrics.f1_macro_test", "metrics.balanced_accuracy_test",
    "attributes.status", "attributes.start_time", "run_id"
]
columnas_disponibles = [col for col in columnas_prioritarias if col in runs.columns]
display(runs[columnas_disponibles])
print(f"Total de runs en el experimento {EXPERIMENT_NAME}: {len(runs)}")

## 10. Conclusiones de P2
El flujo transforma viajes individuales en observaciones estación-hora, define una etiqueta de desequilibrio para la siguiente hora y entrena clasificadores con una partición cronológica. La métrica F1 macro evita que la clase mayoritaria domine por completo la selección, mientras que recall y precisión de la clase `alto` ayudan a interpretar la utilidad del modelo para detectar desequilibrios relevantes.

**Limitaciones:** (1) el flujo neto no mide el inventario real de bicicletas; (2) las horas sin viajes no aparecen como observaciones, por lo que el modelo no predice explícitamente horas sin actividad; (3) los umbrales de riesgo son provisionales; (4) los resultados dependen del periodo y de la cobertura de datos disponibles. Antes de desplegarlo, sería necesario incorporar inventario real, validar umbrales con operación y monitorear cambios en la distribución de datos.